# Preprocesamiento para el modelo

Ingeniería de características sobre las 30 variables definidas en `MODEL_FEATURES` (`pipeline_config.yaml`). Los pasos se aplican en este orden: imputación, codificación de variables categóricas, tratamiento de outliers, transformación de variables continuas y escalado.

El resultado es un `Pipeline` de scikit-learn ajustado solo con el conjunto de entrenamiento, y los datasets transformados que servirán de entrada al notebook de modelos.

## 1. Importación de librerías

Se cargan `feature-engine` para las transformaciones, `scikit-learn` para el pipeline y la división de datos, y la configuración de rutas del proyecto.

In [1]:
import joblib
import numpy as np
import pandas as pd
import yaml
from feature_engine.encoding import OrdinalEncoder
from feature_engine.imputation import AddMissingIndicator, CategoricalImputer, MeanMedianImputer
from feature_engine.outliers import Winsorizer
from feature_engine.transformation import YeoJohnsonTransformer
from feature_engine.wrappers import SklearnTransformerWrapper
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder as OrdinalEncoderSklearn
from sklearn.preprocessing import StandardScaler

from ejercicio_clases.config import MODELS_DIR, PROCESSED_DATA_DIR, PROJ_ROOT, RAW_DATA_DIR

2026-11-03 19:07:34.369 | INFO     | ejercicio_clases.config:<module>:11 - PROJ_ROOT path is: C:\Users\aillu\OneDrive - Empresa Eléctrica de Guatemala, S.A. (EEGSA)\Mis Documentos\Universidad\4. Trimestre\1 - PRODUCT DEVELOPMENT\3. Semana\Ejercicio en Clases\ejercicio_clases


## 2. Parámetros y carga de datos

Se definen las rutas y parámetros del notebook, y se cargan `train.csv`, `test.csv` y la lista de variables `MODEL_FEATURES`. La columna `Id` se usa como índice para conservar la identidad de cada registro.

In [2]:
# Rutas de las fuentes de información (cambiar aquí si los datos se mueven)
TRAIN_FILE = RAW_DATA_DIR / "train.csv"
TEST_FILE = RAW_DATA_DIR / "test.csv"
CONFIG_FILE = PROJ_ROOT / "pipeline_config.yaml"

TARGET = "SalePrice"
TAMANO_VALIDACION = 0.20
SEMILLA = 42

with open(CONFIG_FILE, encoding="utf-8") as archivo:
    MODEL_FEATURES = yaml.safe_load(archivo)["MODEL_FEATURES"]

df_train = pd.read_csv(TRAIN_FILE, index_col="Id")
df_test = pd.read_csv(TEST_FILE, index_col="Id")

print(f"Train: {df_train.shape} | Test: {df_test.shape} | Features: {len(MODEL_FEATURES)}")

Train: (1460, 80) | Test: (1459, 79) | Features: 30


## 3. División en entrenamiento y validación

Se reserva el 20 % de `train.csv` para validación antes de ajustar cualquier transformación, de modo que ese conjunto no influya en el pipeline. `test.csv` no contiene `SalePrice`, por lo que no permite evaluar modelos.

La variable objetivo se transforma con `log1p` para reducir su asimetría (de 1.88 a 0.12). Las predicciones se regresan a dólares con `np.expm1`.

In [3]:
X = df_train[MODEL_FEATURES]
y = np.log1p(df_train[TARGET])
X_test = df_test[MODEL_FEATURES]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=TAMANO_VALIDACION, random_state=SEMILLA
)

print(f"Entrenamiento: {X_train.shape} | Validación: {X_val.shape} | Test: {X_test.shape}")
print(f"Asimetría de {TARGET}: {df_train[TARGET].skew():.2f} -> log1p: {y.skew():.2f}")

Entrenamiento: (1168, 30) | Validación: (292, 30) | Test: (1459, 30)
Asimetría de SalePrice: 1.88 -> log1p: 0.12


## 4. Eliminación de registros atípicos

Se eliminan del conjunto de entrenamiento las 2 viviendas con área habitable mayor a 4,000 y precio muy bajo para su tamaño (`Id` 524 y 1299). Este paso va fuera del pipeline porque un `Pipeline` de scikit-learn no puede eliminar filas. Validación y test no pierden registros.

In [4]:
ids_atipicos = df_train[(df_train["GrLivArea"] > 4000) & (df_train[TARGET] < 300000)].index

print(f"Registros atípicos: {ids_atipicos.tolist()}")
print(f"En entrenamiento: {ids_atipicos.isin(X_train.index).sum()} | En validación: {ids_atipicos.isin(X_val.index).sum()}")

X_train = X_train.drop(ids_atipicos, errors="ignore")
y_train = y_train.drop(ids_atipicos, errors="ignore")

print(f"Entrenamiento final: {X_train.shape}")

Registros atípicos: [524, 1299]
En entrenamiento: 2 | En validación: 0
Entrenamiento final: (1166, 30)


## 5. Grupos de variables

Se clasifican las variables con el mismo criterio del análisis exploratorio: categóricas por tipo de dato, y numéricas como discretas (15 valores únicos o menos) o continuas. Las categóricas se separan además en las que tienen un orden natural y las nominales.

In [5]:
UMBRAL_DISCRETA = 15

variables_categoricas = X_train.select_dtypes(include="str").columns.tolist()
variables_numericas = [var for var in MODEL_FEATURES if var not in variables_categoricas]
variables_continuas = [var for var in variables_numericas if X_train[var].nunique() > UMBRAL_DISCRETA]
variables_discretas = [var for var in variables_numericas if var not in variables_continuas]

# Escalas con orden natural, de menor a mayor
ESCALA_CALIDAD = ["Sin dato", "Po", "Fa", "TA", "Gd", "Ex"]
ESCALA_ACABADO = ["Sin dato", "Unf", "RFn", "Fin"]

ordinales = {
    "ExterQual": ESCALA_CALIDAD,
    "BsmtQual": ESCALA_CALIDAD,
    "KitchenQual": ESCALA_CALIDAD,
    "FireplaceQu": ESCALA_CALIDAD,
    "HeatingQC": ESCALA_CALIDAD,
    "GarageFinish": ESCALA_ACABADO,
}
variables_nominales = [var for var in variables_categoricas if var not in ordinales]

print(f"Continuas ({len(variables_continuas)}): {variables_continuas}")
print(f"Discretas ({len(variables_discretas)}): {variables_discretas}")
print(f"Ordinales ({len(ordinales)}): {list(ordinales)}")
print(f"Nominales ({len(variables_nominales)}): {variables_nominales}")

Continuas (10): ['GrLivArea', 'GarageArea', 'TotalBsmtSF', '1stFlrSF', 'YearBuilt', 'YearRemodAdd', 'GarageYrBlt', 'MasVnrArea', 'BsmtFinSF1', 'LotFrontage']
Discretas (10): ['OverallQual', 'GarageCars', 'FullBath', 'TotRmsAbvGrd', 'Fireplaces', 'HalfBath', 'BsmtFullBath', 'BedroomAbvGr', 'KitchenAbvGr', 'PoolArea']
Ordinales (6): ['ExterQual', 'BsmtQual', 'KitchenQual', 'FireplaceQu', 'HeatingQC', 'GarageFinish']
Nominales (4): ['Neighborhood', 'Foundation', 'GarageType', 'BsmtFinType1']


## 6. Imputación de variables numéricas

Todas las variables numéricas se imputan con la mediana, porque es robusta al sesgo y los extremos. También se incluyen las que solo tienen faltantes en `test.csv`.

Para `LotFrontage`, `GarageYrBlt` y `MasVnrArea`, las únicas con faltantes en `train.csv`, se agrega antes una columna indicadora (`_na`) que conserva la información de que el dato faltaba. En `GarageYrBlt`, los faltantes corresponden a viviendas sin garaje.

In [6]:
variables_con_faltantes = ["LotFrontage", "GarageYrBlt", "MasVnrArea"]

pasos_imputacion_numerica = [
    ("indicador_faltantes", AddMissingIndicator(variables=variables_con_faltantes)),
    ("imputacion_mediana", MeanMedianImputer(imputation_method="median", variables=variables_numericas)),
]

X_imputado_num = Pipeline(pasos_imputacion_numerica).fit_transform(X_train, y_train)

print(f"Faltantes numéricos: {X_train[variables_numericas].isna().sum().sum()} -> {X_imputado_num[variables_numericas].isna().sum().sum()}")
print(f"Columnas nuevas: {[col for col in X_imputado_num.columns if col not in X_train.columns]}")

Faltantes numéricos: 287 -> 0
Columnas nuevas: ['LotFrontage_na', 'GarageYrBlt_na', 'MasVnrArea_na']


## 7. Imputación de variables categóricas

- **`BsmtQual`, `FireplaceQu`, `GarageFinish`, `GarageType` y `BsmtFinType1`:** el vacío indica que la vivienda no tiene la característica, por lo que se asigna la categoría `"Sin dato"`.
- **`KitchenQual`:** el vacío es un dato perdido, ya que todas las viviendas tienen cocina, por lo que se imputa con la categoría más frecuente.

In [7]:
variables_sin_caracteristica = ["BsmtQual", "FireplaceQu", "GarageFinish", "GarageType", "BsmtFinType1"]

pasos_imputacion_categorica = [
    (
        "imputacion_sin_dato",
        CategoricalImputer(imputation_method="missing", fill_value="Sin dato", variables=variables_sin_caracteristica),
    ),
    ("imputacion_moda", CategoricalImputer(imputation_method="frequent", variables=["KitchenQual"])),
]

X_imputado = Pipeline(pasos_imputacion_categorica).fit_transform(X_imputado_num, y_train)

print(f"Faltantes categóricos: {X_train[variables_categoricas].isna().sum().sum()} -> {X_imputado[variables_categoricas].isna().sum().sum()}")
print(f"Faltantes totales tras imputar: {X_imputado.isna().sum().sum()}")

Faltantes categóricos: 731 -> 0
Faltantes totales tras imputar: 0


## 8. Codificación de variables categóricas

- **Ordinales:** se convierten a enteros respetando su orden natural (`Po` < `Fa` < `TA` < `Gd` < `Ex`, y `Unf` < `RFn` < `Fin`), con `"Sin dato"` como nivel más bajo.
- **Nominales (`Neighborhood`, `Foundation`, `GarageType`, `BsmtFinType1`):** se usa `OrdinalEncoder` de feature-engine con `encoding_method="ordered"`, que numera las categorías según su precio promedio en entrenamiento. Una categoría no vista en el ajuste recibe el valor -1.

In [8]:
pasos_codificacion = [
    (
        "codificacion_ordinal",
        SklearnTransformerWrapper(
            OrdinalEncoderSklearn(categories=list(ordinales.values())),
            variables=list(ordinales),
        ),
    ),
    (
        "codificacion_nominal",
        OrdinalEncoder(encoding_method="ordered", variables=variables_nominales, unseen="encode"),
    ),
]

X_codificado = Pipeline(pasos_codificacion).fit_transform(X_imputado, y_train)

print(f"Columnas de texto restantes: {X_codificado.select_dtypes(exclude='number').shape[1]}")
X_codificado[variables_categoricas].head()

Columnas de texto restantes: 0


,Neighborhood,ExterQual,BsmtQual,KitchenQual,GarageFinish,FireplaceQu,Foundation,GarageType,BsmtFinType1,HeatingQC
Id,,,,,,,,,,
255,9,3.0,3.0,3.0,2.0,0.0,2,5,1,3.0
1067,15,4.0,4.0,3.0,2.0,3.0,5,5,5,4.0
639,4,3.0,2.0,3.0,0.0,0.0,2,0,5,4.0
800,8,3.0,4.0,4.0,1.0,3.0,1,2,4,5.0
381,8,3.0,3.0,4.0,1.0,4.0,1,2,2,3.0


## 9. Tratamiento de outliers

Los valores extremos de las variables continuas se limitan con `Winsorizer`, usando el rango intercuartílico (Q1 − 1.5·IQR y Q3 + 1.5·IQR). Los límites se calculan con el conjunto de entrenamiento y no se elimina ningún registro.

In [9]:
pasos_outliers = [
    ("limite_outliers", Winsorizer(capping_method="iqr", tail="both", fold=1.5, variables=variables_continuas)),
]

X_sin_outliers = Pipeline(pasos_outliers).fit_transform(X_codificado, y_train)

pd.DataFrame({
    "valores_limitados": (X_codificado[variables_continuas] != X_sin_outliers[variables_continuas]).sum(),
    "max_antes": X_codificado[variables_continuas].max(),
    "max_despues": X_sin_outliers[variables_continuas].max(),
}).round(1)

,valores_limitados,max_antes,max_despues
GrLivArea,21,4476.0,2757.1
GarageArea,17,1356.0,931.5
TotalBsmtSF,45,3200.0,2050.4
1stFlrSF,12,3228.0,2148.0
YearBuilt,5,2010.0,2010.0
YearRemodAdd,0,2010.0,2010.0
GarageYrBlt,1,2010.0,2010.0
MasVnrArea,77,1378.0,411.9
BsmtFinSF1,3,2096.0,1797.5
LotFrontage,83,313.0,107.5


## 10. Transformación de variables continuas

Se aplica Yeo-Johnson a las 6 variables continuas con asimetría alta en los datos originales (|skew| > 0.75). Este método acepta ceros y encuentra la potencia óptima de cada variable con el conjunto de entrenamiento. Las demás continuas no se transforman.

In [10]:
variables_sesgadas = ["GrLivArea", "TotalBsmtSF", "1stFlrSF", "MasVnrArea", "BsmtFinSF1", "LotFrontage"]

pasos_transformacion = [
    ("yeo_johnson", YeoJohnsonTransformer(variables=variables_sesgadas)),
]

X_transformado = Pipeline(pasos_transformacion).fit_transform(X_sin_outliers, y_train)

pd.DataFrame({
    "skew_original": X_train[variables_sesgadas].skew(),
    "skew_tras_outliers": X_sin_outliers[variables_sesgadas].skew(),
    "skew_final": X_transformado[variables_sesgadas].skew(),
}).round(2)

,skew_original,skew_tras_outliers,skew_final
GrLivArea,0.98,0.58,-0.01
TotalBsmtSF,0.47,0.22,0.05
1stFlrSF,0.80,0.60,-0.00
MasVnrArea,2.29,1.27,0.38
BsmtFinSF1,0.73,0.71,-0.42
LotFrontage,1.71,0.06,0.01


## 11. Escalado de variables

Todas las columnas se llevan a media 0 y desviación estándar 1 con `StandardScaler`. Se usa `SklearnTransformerWrapper` de feature-engine para conservar el formato de DataFrame y los nombres de columna.

In [11]:
pasos_escalado = [
    ("escalado", SklearnTransformerWrapper(StandardScaler())),
]

X_escalado = Pipeline(pasos_escalado).fit_transform(X_transformado, y_train)

X_escalado.describe().loc[["mean", "std"]].T.round(2).head()

,mean,std
GrLivArea,-0.0,1.0
GarageArea,-0.0,1.0
TotalBsmtSF,-0.0,1.0
1stFlrSF,-0.0,1.0
YearBuilt,0.0,1.0


## 12. Pipeline final

Se unen todos los pasos en un único `Pipeline` de scikit-learn y se ajusta solo con el conjunto de entrenamiento.

In [12]:
pipeline_preprocesamiento = Pipeline(
    pasos_imputacion_numerica
    + pasos_imputacion_categorica
    + pasos_codificacion
    + pasos_outliers
    + pasos_transformacion
    + pasos_escalado
)

pipeline_preprocesamiento.fit(X_train, y_train)
pipeline_preprocesamiento

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('indicador_faltantes', ...), ('imputacion_mediana', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0",list,"['Gr...ea', 'Ga...ea', 'To...SF', '1s...SF', ...]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,30
,variables,"['LotFrontage', 'GarageYrBlt', ...]"
,missing_only,True
Name,Type,Value
feature_names_in_,list,"['Gr...ea', 'Ga...ea', 'To...SF', '1s...SF', ...]"


## 13. Generación de datasets

Se transforman los tres conjuntos con el pipeline ajustado y se guardan en `data/processed/`, junto con el pipeline en `models/`. Entrenamiento y validación incluyen el target como `SalePrice_log`; test no lo incluye porque `test.csv` no trae precio.

In [13]:
TARGET_PROCESADO = "SalePrice_log"

train_procesado = pipeline_preprocesamiento.transform(X_train).assign(**{TARGET_PROCESADO: y_train})
val_procesado = pipeline_preprocesamiento.transform(X_val).assign(**{TARGET_PROCESADO: y_val})
test_procesado = pipeline_preprocesamiento.transform(X_test)

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
train_procesado.to_csv(PROCESSED_DATA_DIR / "train_procesado.csv")
val_procesado.to_csv(PROCESSED_DATA_DIR / "validacion_procesado.csv")
test_procesado.to_csv(PROCESSED_DATA_DIR / "test_procesado.csv")

PIPELINE_FILE = MODELS_DIR / "pipeline_preprocesamiento.joblib"
joblib.dump(pipeline_preprocesamiento, PIPELINE_FILE)

print(f"Train: {train_procesado.shape} | Validación: {val_procesado.shape} | Test: {test_procesado.shape}")

Train: (1166, 34) | Validación: (292, 34) | Test: (1459, 33)


## 14. Validación del resultado

Se comprueba que no queden valores faltantes ni columnas de texto, que el pipeline final reproduzca los pasos individuales y que el pipeline guardado y los archivos generados entreguen los mismos resultados.

In [14]:
conjuntos = {"train": train_procesado, "validacion": val_procesado, "test": test_procesado}

for nombre, conjunto in conjuntos.items():
    assert conjunto.isna().sum().sum() == 0, f"{nombre} tiene valores faltantes"
    assert conjunto.select_dtypes(exclude="number").empty, f"{nombre} tiene columnas de texto"

columnas_finales = train_procesado.columns.drop(TARGET_PROCESADO)
assert list(val_procesado.columns.drop(TARGET_PROCESADO)) == list(columnas_finales)
assert list(test_procesado.columns) == list(columnas_finales)

# El pipeline final debe reproducir los pasos ejecutados por separado
pd.testing.assert_frame_equal(train_procesado[columnas_finales], X_escalado[columnas_finales])

# El pipeline guardado debe producir el mismo resultado
pipeline_cargado = joblib.load(PIPELINE_FILE)
pd.testing.assert_frame_equal(pipeline_cargado.transform(X_test), test_procesado)

# El archivo guardado debe conservar los mismos registros y columnas
train_guardado = pd.read_csv(PROCESSED_DATA_DIR / "train_procesado.csv", index_col="Id")
assert train_guardado.shape == train_procesado.shape and train_guardado.index.equals(train_procesado.index)

print(f"Validación correcta: {len(columnas_finales)} columnas de entrada, sin faltantes en los 3 conjuntos.")

Validación correcta: 33 columnas de entrada, sin faltantes en los 3 conjuntos.
